# SIH26142 — Deep Super-Resolution Fine-Tuning (Google Colab)
### Sentinel-2 ×4 Super-Resolution | Real-ESRGAN (RRDBNet Backbone)

**What this notebook does:**
1. Checks GPU, installs dependencies
2. Clones the project from GitHub
3. Verifies / generates synthetic training pairs
4. Runs multi-loss fine-tuning (L1 + Perceptual + SAM + Edge + FFT)
5. Plots training curves
6. Downloads the best checkpoint back to this machine

> **Recommended runtime:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — GPU Check
Must show a CUDA GPU. If not, change runtime to T4 GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.')


## Cell 2 — Install Dependencies
Installs basicsr, realesrgan, rasterio and supporting packages.

In [ ]:
# Fix basicsr/torchvision compatibility shim (must come before basicsr imports)
import torchvision.transforms.functional as _F_t
import sys
sys.modules['torchvision.transforms.functional_tensor'] = _F_t

!pip install -q basicsr facexlib gfpgan realesrgan rasterio scikit-image \
    tqdm opencv-python matplotlib
print('All packages installed.')


## Cell 3 — Clone Repository
Clones your GitHub repo. **Edit the URL below** if your repo path differs.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/AjayBora002/Depth-Wizard.git'  # <-- your repo
CLONE_DIR = '/content/Depth-Wizard'
PROJECT_DIR = f'{CLONE_DIR}/srm-project'

if not os.path.exists(CLONE_DIR):
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {CLONE_DIR} pull

# Add project root to path so 'src' is importable
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls src/


## Cell 4 — Download Pretrained Weights
Auto-downloads `RealESRGAN_x4plus.pth` into `src/checkpoints/` if not already present.

In [ ]:
from src.model import _download_weights
weight_path = _download_weights('x4plus')
print('Weights ready at:', weight_path)


## Cell 5 — Generate Synthetic Training Pairs
If pairs already exist in `data/synthetic_pairs/`, this step is skipped.
Otherwise it synthesises LR/HR pairs from the Sentinel-2 tiles in `data/raw/`.

> **Note:** The repo should already contain sample `.npy` pairs committed to `data/synthetic_pairs/`.
> If not, you need to either upload tiles to `data/raw/` and run generation,
> or upload the pairs directly.

In [ ]:
import os
from pathlib import Path

lr_dir = Path('data/synthetic_pairs/lr')
hr_dir = Path('data/synthetic_pairs/hr')
n_lr = len(list(lr_dir.glob('*.npy'))) if lr_dir.exists() else 0
n_hr = len(list(hr_dir.glob('*.npy'))) if hr_dir.exists() else 0
print(f'Found {n_lr} LR pairs and {n_hr} HR pairs')

if n_lr == 0 or n_hr == 0:
    raw_tiles = list(Path('data/raw').glob('*.tif'))
    if not raw_tiles:
        print('WARNING: No tiles in data/raw/ and no pairs found.')
        print('Upload .tif tiles to data/raw/ via Files panel, then re-run this cell.')
    else:
        print(f'Generating pairs from {len(raw_tiles)} tile(s)...')
        from src.pair_generation import generate_all_pairs
        generate_all_pairs(
            raw_dir='data/raw',
            out_dir='data/synthetic_pairs',
            scale=4,
            patches_per_tile=50,
        )
        print('Pair generation complete.')
else:
    print(f'Pairs already present ({n_lr} LR / {n_hr} HR). Skipping generation.')


## Cell 6 — Fine-Tune the Model
Runs multi-loss training: **L1 + Perceptual (VGG16) + SAM + Edge + FFT**.

Typical Colab T4 speed: **~1–2 min/epoch** for `crop_size=128, batch_size=4`.
50 epochs ≈ 50–100 minutes total.

The `time_budget_hours=2` cap is a safety net — adjust or remove as needed.

In [ ]:
from src.train import train

results = train(
    pairs_dir='data/synthetic_pairs',
    output_dir='src/checkpoints',
    model_key='x4plus',
    epochs=50,
    batch_size=4,
    crop_size=128,
    lr=1e-4,
    lambda_perceptual=0.1,
    lambda_sam=0.05,
    lambda_edge=0.05,
    lambda_freq=0.02,
    save_every=10,
    num_workers=2,
    use_amp=True,           # AMP (FP16) for T4 speed boost
    time_budget_hours=2.0, # Stop cleanly if Colab session nears timeout
)

print('\nTraining complete!')
print('Best checkpoint:', results['best_checkpoint'])
hist = results['history']
print(f'Epochs run: {len(hist["train_loss"])}')
print(f'Best Val PSNR: {max(hist["val_psnr"]):.2f} dB')
print(f'Total training time: {sum(hist["epoch_time"])/60:.1f} min')


## Cell 7 — Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('src/checkpoints/training_history.json') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, hist['train_loss'], 'b-o', markersize=4, label='Train')
axes[0].plot(epochs, hist['val_loss'], 'r-o', markersize=4, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist['val_psnr'], 'g-o', markersize=4)
axes[1].set_title('Validation PSNR (dB)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, hist['epoch_time'], 'm-o', markersize=4)
axes[2].set_title('Epoch Time (s)')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.suptitle('SIH26142 — SR Fine-Tuning Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('src/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curves saved to src/checkpoints/training_curves.png')


## Cell 8 — Download Checkpoints
Downloads the best checkpoint, training history, and curves to your local machine.

> After downloading, place `model_finetuned_best.pth` in your local
> `srm-project/src/checkpoints/` folder to use it for inference.

In [ ]:
from google.colab import files
import os

to_download = [
    'src/checkpoints/model_finetuned_best.pth',
    'src/checkpoints/training_history.json',
    'src/checkpoints/training_curves.png',
]

for path in to_download:
    if os.path.exists(path):
        print(f'Downloading {path}...')
        files.download(path)
    else:
        print(f'SKIP (not found): {path}')

print('Done. Place model_finetuned_best.pth in srm-project/src/checkpoints/ locally.')
